## Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

## Concatenating PLACES Datasets

[CDC PLACES Datasets](https://data.cdc.gov/browse?category=500+Cities+%26+Places&q=PLACES%3A+Local+Data+for+Better+Health%2C+County+Data&sortBy=relevance&tags=places&pageSize=20)

The CDC provides annual releases of the PLACES dataset from 2020 - 2025. One important thing to note is that each release sources data from the Behavioral Risk Factor Surveillance System (BRFSS) two years prior, so the true chronological range of our PLACES dataset is 2018 - 2023. 

Another important thing to note is that from 2022 - 2023, the relevant measure we are investigating is labelled **"Frequent mental distress among adults"**, while from 2018 - 2021 it is labelled **"Mental health not good for >=14 days among adults aged >=18 years"** After a quick Google search, BRFSS treats them as the same thing, and as you'll see in the code blocks below, they share the same **MeasureId**.

For simplicity, the link to the datasets provided also have a query tool, where we can essentially filter the raw dataset to our choosing. These are the filters I put in place for each annual dataset prior to exporting to a CSV:
- **MeasureId** is equal to **MHLTH**
- **DataValueTypeID** is equal to **AgeAdjPrv**
- **StateAbbr** is NOT **"US"**

In [ ]:
places_2023_df = pd.read_csv("Data/PLACES/PLACES_2023.csv")
places_2023_df

In [ ]:
places_2019_df = pd.read_csv("Data/PLACES/PLACES_2019.csv")
places_2019_df

In [ ]:
extracted_columns = ["Year", "StateDesc", "LocationName", "Data_Value", "Low_Confidence_Limit", "High_Confidence_Limit", "TotalPopulation", "Geolocation"]

years = np.arange(2018, 2024)
places_df = pd.DataFrame(columns=extracted_columns)
for year in years:
    file_name = f"Data/PLACES/PLACES_{year}.csv"
    print(f"Reading {file_name} ...")
    curr_places_df = pd.read_csv(file_name)
    places_df = pd.concat([places_df, curr_places_df[extracted_columns]], ignore_index=True)
    print("Complete!")
places_df = places_df.rename(columns={"StateDesc": "State", "LocationName": "County", "Data_Value": "FMD_Percent"})

In [ ]:
print(f"Total rows: {len(places_df)}")
print(f"Years covered: {places_df['Year'].unique()}")
print(f"States covered: {places_df['State'].unique()}")

In [ ]:
places_df.to_csv("Data/Processed/PLACES_Combined.csv", index=False)

# Supervised Learning

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
df = pd.read_csv("data/processed/county_panel_imputed.csv")
print("Min FMD:", df['mental_health_prevalence'].min())
print("Max FMD:", df['mental_health_prevalence'].max())
df.head()

In [ ]:
features = [
    #'Good Days', 
    #'Moderate Days', 
    #'Unhealthy for Sensitive Groups Days', 
    #'Unhealthy Days', 
    #'Very Unhealthy Days', 
    #'Hazardous Days', 
    'Median AQI', 
    'Days CO', 
    'Days NO2', 
    'Days Ozone', 
    'Days PM2.5', 
    'Days PM10',
    'median_income', 
    'poverty_count', 
    'population', 
    'unemployed_count']

X = df[features]
y = df["mental_health_prevalence"]

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,      
    random_state=42,     
    shuffle=True         
)

## Ridge Regression

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge())
])

param_grid = {
    "ridge__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best alpha:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

In [ ]:
best_ridge = grid.best_estimator_
y_pred = best_ridge.predict(X_test)

print("Test RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test R^2:", r2_score(y_test, y_pred))

## Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
)

rf.fit(X_train, y_train)

In [ ]:
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(importance)

In [ ]:
y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False) 
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R² Score (Coefficient of Determination): {r2:.4f}")

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", 0.5, 0.8],
    "bootstrap": [True, False]
}

rf = RandomForestRegressor(random_state=42)

rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,                      # number of parameter combinations to try
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=2,
    random_state=42
)

rf_random.fit(X_train, y_train)

print("Best params:", rf_random.best_params_)
print("Best CV RMSE:", -rf_random.best_score_)

Best params: \{'n\_estimators': 300, 'min\_samples\_split': 2, 'min\_samples\_leaf': 1, 'max\_features': 0\.5, 'max\_depth': None, 'bootstrap': False\}
Best CV RMSE: 1\.708676959054807